# Interactive plots - The `bloch_schrodinger.plotting` module

This tutorial goes through the interactive plotting functions of the package on a single example, an optical honeycomb lattice with a tunable depth and sublattice imbalance. We will plot:

1. potential landscapes, and cuts through them,
2. eigenvalue landscapes (band structures, and quantities derived from them),
3. eigenvectors, and cuts through them,
4. both at the same time,

and finally show how to style all of these plots with templates.

All the plotting functions work the same way: **every dimension of the data that is not plotted becomes a slider.** You never build the sliders yourself. Whatever parameters you created with `create_parameter`, and whatever k-points you solved on, show up as sliders automatically.

The spatial axes to plot against are chosen with `cart_axes`, where `0 = x`, `1 = y` and `2 = z`. Two axes give a 2D map, one axis gives a cut, and any spatial axis left out becomes a slider as well. That slider sets where the cut sits.

## The honeycomb lattice

We use the same lattice as in the [Wannier](Wannier.ipynb) tutorial: three laser beams at 120° create a honeycomb lattice. The lattice depth `s` and the sublattice imbalance `delta` are parameters, so each of them gets a slider in every plot below.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from bloch_schrodinger.potential import Potential, create_parameter

# Creating the optical lattice, see https://journals.aps.org/pra/abstract/10.1103/PhysRevA.80.043411 for more information on the lattice.
## The lattice parameters
kl = 2
m = 1
hbar = 1
E_r = hbar**2 * kl**2 / 2 / m
delta = create_parameter('delta', np.linspace(-0.2, 0.2, 5)) # sublattice imbalance
s = create_parameter('s', np.linspace(3, 9, 3)) # lattice depth, in recoil energies

V1 = -(s - delta) * E_r
V2 = -(s + delta) * E_r

## The lattice geometry
a = 4*np.pi / 3**1.5 / kl
a1 = np.array([3 * a/2, -(3**0.5) * a/2])  # 1st lattice vector
a2 = np.array([3 * a/2, 3**0.5 * a/2])  # 2nd lattice vector

k1 = kl * np.array([-3**0.5 / 2 , 1/2])
k2 = kl * np.array([3**0.5 / 2 , 1/2])
k3 = kl * np.array([0,-1])

K = np.array([0, 4*np.pi/3**1.5/a]) # The K point of the Brillouin zone

## Creating the potential object
honeycomb = Potential(
    unitvecs = [a1, a2],
    resolution = (48, 48),
    v0 = 0,
)

dirs = [k[0] * (honeycomb.x - a1[0]) + k[1] * honeycomb.y for k in (k1, k2, k3)]

for i in range(3):
    honeycomb.add(2*V1*np.cos((dirs[i-1]-dirs[i]) - 2*np.pi/3)/2)
    honeycomb.add(2*V2*np.cos((dirs[i-1]-dirs[i]) + 2*np.pi/3)/2)

print(honeycomb.V.dims)

## 1. Potential landscapes

The quickest option is the `plot` method of the `Potential` class. `show_minima=True` marks the local minima of the potential, here the two lattice sites A and B of the unit cell. Move the `delta` slider to see one site get deeper than the other.

In [ ]:
honeycomb.plot(show_minima=True)
plt.show()

A single unit cell is not very telling. `tile` returns a copy of the potential repeated over several unit cells, which is much easier to read. The two pairs give the range of cells along `a1` and `a2`.

In [ ]:
tiled = honeycomb.tile([[-1, 2], [-1, 2]])
tiled.plot()
plt.show()

For more control over the style, pass the potential's DataArray `.V` to `plot_eigenvector`. This is the package's general plotting function for fields, which we will use for the eigenvectors below. It takes **matrices** (lists of lists) of fields, of potentials drawn as contour overlays, and of templates. Each entry of these matrices is one subplot of the figure.

The templates set the style of each subplot. For now, it is enough to know that a name such as `'real'` picks a preset style, and a tuple `(colormesh, contour, quiver)` styles the three layers of a subplot separately. [Section 7](#7.-Styling-plots-with-templates) covers templates in detail.

Here the left panel shows the potential with 6 dashed contour levels. The right panel has no colormesh (`None` in `plots`): it shows only the contours, with the optical dipole force $\mathbf{F} = -\nabla V$ overlaid as arrows through the `quivers` argument. The force is computed from the derivatives along the lattice axes $a_1$ and $a_2$, converted to cartesian components.

In [ ]:
from bloch_schrodinger.plotting import plot_eigenvector, contour_tmpl

# dV/da_i = a_i . grad(V), so grad(V) = A^-1 (dV/da_1, dV/da_2), with A the matrix of the lattice vectors
A_inv = np.linalg.inv(np.array([a1, a2]))
dV1 = tiled.V.differentiate('a1')
dV2 = tiled.V.differentiate('a2')
Fx = -(A_inv[0, 0] * dV1 + A_inv[0, 1] * dV2)
Fy = -(A_inv[1, 0] * dV1 + A_inv[1, 1] * dV2)

force_template = {
    # passed to plt.quiver: with scale_units='xy', an arrow of length 1 represents a force of 'scale'
    'fkwargs': {'color': 'C3', 'pivot': 'mid', 'angles': 'xy', 'scale_units': 'xy', 'scale': 120, 'width': 0.006},
    'density': 12, # only one arrow every 12 grid points
}

fig, axs = plot_eigenvector(
    plots = [[tiled.V, None]], # None: no colormesh in the second panel, only contours and arrows
    potentials = [[tiled, tiled]],
    templates = [[('real', contour_tmpl(6)), (None, contour_tmpl(6), force_template)]],
    quivers = [[None, (Fx, Fy)]],
)
axs[0][0].set_title('$V(x, y)$')
axs[0][1].set_title(r'$\mathbf{F} = -\nabla V$')
plt.show()

## 2. Cuts through the potential

Giving a single axis to `cart_axes` plots a cut instead of a map. The other spatial axis becomes a slider: with `cart_axes=[0]` the cut runs along `x`, and the `y` slider moves it across the lattice.

In [ ]:
tiled.plot(cart_axes=[0])
plt.show()

`plot_eigenvector` accepts the same argument, either as a single list for every subplot, or as a matrix of the same shape as `plots` to give each subplot its own axes. Here we draw a cut along `x` next to a cut along `y`.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[tiled.V, tiled.V]],
    potentials = [[None, None]],
    cart_axes = [[[0], [1]]],
)
axs[0][0].set_title('cut along x')
axs[0][1].set_title('cut along y')
plt.show()

## Solving the band structure

We now compute the bands along a line through the Brillouin zone, from $-1.5K$ to $1.5K$ along $k_y$. This line passes through $\Gamma$ and both Dirac points $\pm K$. With a single `kx` value the band structure is a function of `ky`, `s` and `delta`.

That is 555 diagonalizations (37 k-points × 3 depths × 5 imbalances), so we parallelize the solve.

In [ ]:
from bloch_schrodinger.fdsolver import FDSolver

solver = FDSolver(honeycomb, alphas = hbar**2/2/m)
solver.create_reciprocal_grid([0, np.linspace(-1.5*K[1], 1.5*K[1], 37)])

eigva, eigve = solver.solve(n_eigva = 4, parallel = True)

print(eigva.dims) # kx only takes a single value, so it is not a dimension (and won't get a slider)
print(eigve.dims)

## 3. Eigenvalue landscapes

`plot_cuts` plots a DataArray against one of its dimensions, and draws one line per value of the dimensions listed in `groupby` (`['band']` by default). Here that gives the band structure along `ky`, with `s` and `delta` as sliders. At `delta = 0` the two lowest bands touch at the Dirac points $\pm K$ (dotted lines). Any imbalance opens a gap there.

In [ ]:
from bloch_schrodinger.plotting import plot_cuts

fig, ax = plot_cuts(eigva, 'ky')
for k in [-K[1], K[1]]:
    ax.axvline(k, color='gray', linestyle='dotted')
plt.show()

The styles in `linekws` are cycled over the bands, and `ymin`/`ymax` fix the vertical range, which otherwise follows the sliders. Selecting before plotting reduces the number of lines and sliders: here we keep only the two lowest bands.

In [ ]:
fig, ax = plot_cuts(
    eigva.sel(band=[0, 1]), 'ky',
    linekws = [{'color': 'C0'}, {'color': 'C3', 'linestyle': '--'}],
    ymin = -20, ymax = 5,
)
plt.show()

`plot_cuts` works on any DataArray, not only on the raw eigenvalues. Quantities derived with ordinary xarray operations are plotted the same way. Here the gap at the Dirac point is plotted against `delta` with `groupby=[]`, since it is a single curve, and the width of the lowest band against `s`.

In [ ]:
gap_K = (eigva.sel(band=1) - eigva.sel(band=0)).sel(ky=K[1], method='nearest').rename('gap at K')
fig, ax = plot_cuts(gap_K, 'delta', groupby=[])
plt.show()

bandwidth = (eigva.max('ky') - eigva.min('ky')).rename('bandwidth')
fig, ax = plot_cuts(bandwidth, 's')
plt.show()

## 4. Eigenvectors

The eigenvectors are plotted with `plot_eigenvector`, exactly like the potential above. Plot real quantities: the density `abs(eigve)**2`, the real part `eigve.real`, or the phase. For the phase, use `xr.ufuncs.angle(eigve)` rather than `np.angle`, which returns a bare numpy array that lost its dimensions and coordinates.

Every other dimension becomes a slider: the band, the k-point and the lattice parameters. Move `ky` to the K point (`ky ≈ 2.0`) to see the phase wind around the lattice sites.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[abs(eigve)**2, eigve.real, xr.ufuncs.angle(eigve)]],
    potentials = [[honeycomb, honeycomb, honeycomb]],
    templates = [['amplitude', 'real', 'phase']],
)
axs[0][0].set_title(r'$|\psi|^2$')
axs[0][1].set_title(r'Re$(\psi)$')
axs[0][2].set_title(r'arg$(\psi)$')
plt.show()

## 5. Cuts through the eigenvectors

As for the potential, a single cartesian axis gives a cut, with the other axis on a slider. The potential given in `potentials` is drawn dashed, against its own y-axis on the right, since both quantities have very different scales. A `cart_axes` matrix mixes maps and cuts in the same figure.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[abs(eigve)**2, abs(eigve)**2]],
    potentials = [[honeycomb, honeycomb]],
    templates = [['amplitude', None]],
    cart_axes = [[[0, 1], [0]]],
)
plt.show()

## 6. Eigenvalues and eigenvectors together

`dashboard` puts the band structure next to maps of the eigenvectors, all driven by the same sliders. A red dashed line marks the current position along the dimension the bands are plotted against, here `ky`. The `eigveplots` matrix sets the layout of the maps, like `plots` in `plot_eigenvector`. A single `template` styles all the maps, see [section 7](#Styling-the-dashboard).

In [ ]:
from bloch_schrodinger.plotting import dashboard

fig, ax = dashboard(
    eigva, 'ky',
    eigveplots = [[abs(eigve)**2]],
    potential = honeycomb,
    template = 'amplitude',
    titles = [[r'$|\psi|^2$']],
)
plt.show()

`energy_levels` draws a cut through the potential, and places the profile of each mode along that cut at its energy. The `offset` and `cut rotation` sliders move the cut line in the plane. Passing the tiled potential shows the lattice around the unit cell. The eigenvectors are only defined over a single unit cell, so their profiles stop at its edges.

In [ ]:
from bloch_schrodinger.plotting import energy_levels

fig, ax = energy_levels(eigva, eigve.real, tiled)
plt.show()

## 7. Styling plots with templates

Templates are how the style of `plot_eigenvector` plots is controlled. **A template is a plain dictionary** of instructions for one layer of a subplot. Each subplot of `plot_eigenvector` has up to three layers, each with its own template:

- the **colormesh** of the field given in `plots`, drawn with `pcolormesh`,
- the **contours** of the potential given in `potentials`, drawn with `contour`,
- the **arrows** given in `quivers`, drawn with `quiver`.

Each entry of the `templates` matrix sets the templates of one subplot. It can take the following forms:

| Entry | Colormesh | Contours | Arrows |
|---|---|---|---|
| `None` | matplotlib defaults, no colorbar | matplotlib defaults | matplotlib defaults |
| `'amplitude'` (a preset name) | `cmesh_tmpl('amplitude')` | `contour_tmpl(ncontours)` | `quiver_tmpl()` |
| `my_dict` | `my_dict` | `contour_tmpl(ncontours)` | `quiver_tmpl()` |
| `(c,)` | `c`, a name or a dict | `contour_tmpl(ncontours)` | `quiver_tmpl()` |
| `(c, ct)` | `c` | `ct`, a dict | `quiver_tmpl()` |
| `(c, ct, q)` | `c` | `ct` | `q`, a dict |

The functions `cmesh_tmpl`, `contour_tmpl` and `quiver_tmpl` return the default templates, so printing them shows what a template contains:

In [ ]:
from bloch_schrodinger.plotting import cmesh_tmpl, contour_tmpl, quiver_tmpl

print(cmesh_tmpl('amplitude'))
print(contour_tmpl())
print(quiver_tmpl())

### The keys of a template

**Colormesh templates** (the first entry):
- `fkwargs`: keyword arguments passed to `pcolormesh`, e.g. `cmap`, `norm`, `shading` or `vmin`/`vmax`. A `norm` can be given as a function returning a norm, e.g. `lambda: colors.LogNorm()`. Each subplot then builds its own norm, instead of all of them sharing, and rescaling, the same one.
- `autoscale`: if `True`, the color range follows the data each time a slider moves. If `False`, it stays where it started.
- `clim`: a pair `(vmin, vmax)` fixing the color range.
- `colorbar`: without this key, or with `None`, there is no colorbar. Otherwise, a dictionary with the optional keys:
    - `kwargs`, passed to `Figure.colorbar` (e.g. `label`, `format`, `orientation`),
    - `cax`, the position of the colorbar, passed to `append_axes` (default `{'position': 'right', 'size': '5%', 'pad': 0.05}`),
    - `ticks` and `tickslabel`, to set the ticks and their labels by hand.
- `slider_start`: where the sliders start, `'left'` (default), `'mid'` or `'right'`.

**Contour templates** (the second entry): `fkwargs` is passed to `contour`, e.g. `levels`, `colors` or `cmap`, `linewidths` and `linestyles`. The `ncontours` argument of `plot_eigenvector` is a shortcut that sets the number of levels of the default contour template.

**Quiver templates** (the third entry): `fkwargs` is passed to `quiver`, and `density` keeps only one arrow every `density` grid points.

**Line templates**, for cuts (`cart_axes` with a single axis): only `fkwargs`, passed to `plot`, `autoscale` (of the y-axis, default `True`) and `slider_start` are used. The presets are made for maps: their `cmap` and `norm` keywords make `plot` fail, so style cuts with a dictionary or `None`.

Two more pitfalls:
- The contour and quiver entries of a tuple must be dictionaries, not preset names or `None`. To draw the contours with matplotlib's defaults, give `{}`. To hide them, give `None` in `potentials` instead.
- `get_template` returns templates in an older format, which `plot_eigenvector` does not read. Use `cmesh_tmpl` instead.

### The presets

The five presets returned by `cmesh_tmpl` are:
- `'amplitude'`: a sequential colormap on a linear scale, for positive fields such as a density.
- `'amplitude - log'`: the same on a log scale, to see the tails and the nodes of a mode.
- `'real'`: a diverging colormap centered on zero, for signed fields such as the real part of a mode.
- `'real - log'`: the same on a symmetric log scale.
- `'phase'`: a cyclic colormap fixed to $[-\pi, \pi]$, with ticks labelled in units of $\pi$.

Here are four of them on the same eigenvector. The log scale reveals the node of the wavefunction between the two sites, which is invisible on the linear scale. `'real - log'` gets its own example below.

In [ ]:
density = abs(eigve)**2

fig, axs = plot_eigenvector(
    plots = [[density, density], [eigve.real, xr.ufuncs.angle(eigve)]],
    potentials = [[honeycomb, honeycomb], [honeycomb, honeycomb]],
    templates = [['amplitude', 'amplitude - log'], ['real', 'phase']],
)
for ax, title in zip(np.ravel(axs), ["'amplitude'", "'amplitude - log'", "'real'", "'phase'"]):
    ax.set_title(title)
plt.show()

### Modifying a preset

Since presets are plain dictionaries, the simplest way to get a custom style is to take a preset and change some of its keys.

On the left, we plot the real part of the states at the $\Gamma$ point with `'real - log'`. By default, this preset has a linear region of only $10^{-12}$ around zero, so its colorbar gets ticks for every decade down to $10^{-12}$. Here we widen the linear region with a new norm, given as a function. The log scale shows the small tails of the wavefunction much more clearly than `'real'` does. Move the `band` slider to see the sign changes of the higher bands.

On the right, we give `'amplitude'` another colormap and a label. We also fix its color range and turn autoscaling off, so that densities can be compared between bands and k-points while moving the sliders.

In [ ]:
from matplotlib import colors

real_log = cmesh_tmpl('real - log')
real_log['fkwargs']['norm'] = lambda: colors.SymLogNorm(linthresh=1e-2)

fixed_density = cmesh_tmpl('amplitude')
fixed_density['fkwargs']['cmap'] = 'magma'
fixed_density['autoscale'] = False
fixed_density['clim'] = (0, 3)
fixed_density['colorbar']['kwargs'] = {'label': r'$|\psi|^2$', 'format': '{x:.1f}'}

psi_gamma = eigve.sel(ky=0, method='nearest') # the states at the Gamma point

fig, axs = plot_eigenvector(
    plots = [[psi_gamma.real, density]],
    potentials = [[honeycomb, honeycomb]],
    templates = [[real_log, fixed_density]],
)
plt.show()

### Writing a template from scratch

A template only needs the keys you want to set. Here the field is drawn with a matplotlib colormap and smooth `gouraud` shading. The colorbar is horizontal, below the map, and the sliders start in the middle of their range.

In [ ]:
smooth_real = {
    'fkwargs': {'cmap': 'RdBu_r', 'shading': 'gouraud', 'norm': lambda: colors.CenteredNorm()},
    'autoscale': True,
    'colorbar': {
        'kwargs': {'orientation': 'horizontal', 'label': r'Re$(\psi)$'},
        'cax': {'position': 'bottom', 'size': '5%', 'pad': 0.3},
    },
    'slider_start': 'mid',
}

fig, axs = plot_eigenvector(
    plots = [[eigve.real]],
    potentials = [[honeycomb]],
    templates = [[smooth_real]],
)
plt.show()

### Styling the contours and the arrows

The second entry of a tuple styles the contours of the potential. `contour_tmpl` accepts either a number of levels or the levels themselves. A dictionary gives full control, here with contours colored by a colormap. The arrows are styled the same way through the third entry, as done for the force field in section 1.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[density, density]],
    potentials = [[honeycomb, honeycomb]],
    templates = [[
        ('amplitude', contour_tmpl(np.linspace(-15, 30, 10))), # 10 levels at fixed energies
        ('amplitude', {'fkwargs': {'levels': 8, 'cmap': 'inferno', 'linewidths': 1}}),
    ]],
)
plt.show()

### Styling cuts

Cuts are styled with line templates. A matrix of templates and a matrix of `cart_axes` give each subplot its own style. Here a map with a preset is next to a cut, drawn as a thick red line, whose y-axis stays fixed while the sliders move.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[density, density]],
    potentials = [[honeycomb, honeycomb]],
    templates = [['amplitude', {'fkwargs': {'color': 'C3', 'linewidth': 2}, 'autoscale': False}]],
    cart_axes = [[[0, 1], [0]]],
)
plt.show()

### Styling the dashboard

`dashboard` takes one template, shared by all its maps. It accepts a preset name, a colormesh template dict, or a tuple `(colormesh, contour)` to style the potential contours as well. There is no quiver entry, since the dashboard draws no arrows. The maps never get a colorbar, since they share the sliders but not their color scales.

In [ ]:
fig, ax = dashboard(
    eigva, 'ky',
    eigveplots = [[density, abs(psi_gamma)**2]], # the second map does not follow the ky slider
    potential = honeycomb,
    template = (fixed_density, contour_tmpl(8)), # the fixed color range makes both maps comparable
    titles = [[r'$|\psi|^2$', r'$|\psi|^2$ at $\Gamma$']],
)
plt.show()

## Summary

| What you want | Call |
|---|---|
| Potential landscape | `pot.plot()`, or `plot_eigenvector([[pot.V]], [[None]], [['real']])` |
| Cut through the potential | `pot.plot(cart_axes=[0])`, or `plot_eigenvector(..., cart_axes=[0])` |
| Band structure, eigenvalues vs. a parameter | `plot_cuts(eigva, 'ky')` |
| Any derived quantity vs. a parameter | `plot_cuts(quantity, 'delta', groupby=[])` |
| Eigenvector maps | `plot_eigenvector([[abs(eigve)**2]], [[pot]], [['amplitude']])` |
| Cuts through eigenvectors | `plot_eigenvector(..., cart_axes=[0])` |
| Bands and eigenvectors side by side | `dashboard(eigva, 'ky', [[abs(eigve)**2]], pot, 'amplitude')` |
| Mode profiles at their energies | `energy_levels(eigva, eigve, pot)` |
| Style a subplot | `templates=[['amplitude']]`, `[[(cmesh_tmpl('real'), contour_tmpl(6), quiver_tmpl())]]`, or dictionaries |
| Style a dashboard | `template='amplitude'`, a dict, or `(colormesh, contour)` |

Every function returns the matplotlib figure and axes, so titles, limits and labels can be adjusted afterwards. For plots that none of these functions cover, the low-level `create_map`, `create_line` and `create_quiver` functions handle the interactivity of a single panel of your own figure. See the [Wannier](Wannier.ipynb) and [PolaritonDispersion](PolaritonDispersion.ipynb) tutorials.